# CASMI 2026 — V1: library search + database retrieval + de novo generation

For every unknown molecule the adduct-aware neutral mass (±10 ppm, instrument bias calibrated) defines a candidate set from three sources:

1. **training structures** ranked by spectral similarity to their public library spectra (class 1),
2. **COCONUT + PubChem** structures in the mass window (class 2),
3. **de novo samples** from a spectrum→SMILES transformer, kept only if their mass matches (class 3).

One transformer encoder feeds a Morgan-fingerprint head (ranks database candidates) and a SMILES decoder (candidate likelihood + generation). A small linear ranker fitted on held-out molecules orders the merged candidates; the top 25 unique tautomer-canonical InChIKey blocks are submitted.

**Inputs:** the competition data, plus the dataset holding `model.pt`, `vocab.json`, `ranker.json`, `structures.parquet`, `coconut.parquet`, `pubchem.parquet` (all built from public data: the competition training set, COCONUT (CC0) and PubChem).

In [ ]:
import os, sys, glob, subprocess

COMP_DIR = next(iter(glob.glob('/kaggle/input/**/enveda-CASMI26-molecule-id-mass-spectra', recursive=True)), '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra')
hits = glob.glob('/kaggle/input/**/model.pt', recursive=True)
assert hits, 'Attach the CASMI V1 assets dataset (model.pt, vocab.json, structures.parquet, coconut.parquet, ...) to this notebook'
ASSETS_DIR = os.path.dirname(hits[0])
print('competition data:', COMP_DIR, os.listdir(COMP_DIR))
print('assets          :', ASSETS_DIR, os.listdir(ASSETS_DIR))

# RDKit: use the environment's copy if present (Dependency Manager: rdkit==2026.3.3), else install the bundled wheel offline.
try:
    import rdkit
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--find-links', ASSETS_DIR, 'rdkit'])
    import rdkit
print('rdkit', rdkit.__version__)
os.makedirs('/kaggle/working/casmi', exist_ok=True)
open('/kaggle/working/casmi/__init__.py', 'w').close()
os.chdir('/kaggle/working')
sys.path.insert(0, '/kaggle/working')

In [ ]:
%%writefile casmi/common.py
"""
Shared core for the CASMI 2026 pipeline: adduct arithmetic, the Kaggle metric key,
spectrum preprocessing, SMILES tokenisation and fingerprints.

Pure python + numpy + rdkit so the same file runs locally and inside the Kaggle notebook.
"""

import re
import json
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Descriptors import ExactMolWt
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
import rdkit.RDLogger as _rkl
import rdkit.rdBase as _rkrb

_rkl.logger().setLevel(_rkl.ERROR)
_rkrb.DisableLog("rdApp.*")

# ─────────────────────────────────────────────────────────────────────
# Adduct arithmetic
# ─────────────────────────────────────────────────────────────────────

ELECTRON_MASS = 0.00054857990907
_PT = Chem.GetPeriodicTable()

# The ten adducts the competition says the hidden test set uses.
TEST_ADDUCTS = [
    '[M+H]+', '[M+NH4]+', '[M-H2O+H]+', '[M-2H2O+H]+', '[M+Na]+', '[M+K]+',
    '[M-H]-', '[M-H2O-H]-', '[M+CH2O2-H]-', '[M+Cl]-',
]
ADDUCT_TO_ID = {a: i + 1 for i, a in enumerate(TEST_ADDUCTS)}  # 0 = unknown/other

_FORMULA_TOKEN = re.compile(r'([A-Z][a-z]?)(\d*)')
_ADDUCT_RE = re.compile(r'^\[(\d*)M([^\]]*)\](\d*)([+-])$')
_ADDUCT_PART = re.compile(r'([+-])(\d*)([A-Za-z][A-Za-z0-9]*)')


def formula_mass(formula):
    """Monoisotopic mass of a plain formula string such as 'CH2O2'."""
    total = 0.0
    for sym, count in _FORMULA_TOKEN.findall(formula):
        total += _PT.GetMostCommonIsotopeMass(sym) * (int(count) if count else 1)
    return total


_adduct_cache = {}


def parse_adduct(adduct):
    """'[2M+Na-2H]-' -> (n_mol, mass_shift, charge) with m/z = (n_mol*M + mass_shift) / |charge|.

    The shift includes the electron-mass correction. Returns None when the string cannot be parsed.
    """
    if adduct in _adduct_cache:
        return _adduct_cache[adduct]
    result = None
    m = _ADDUCT_RE.match(adduct.replace(' ', '')) if isinstance(adduct, str) else None
    if m:
        n_mol = int(m.group(1)) if m.group(1) else 1
        charge = (int(m.group(3)) if m.group(3) else 1) * (1 if m.group(4) == '+' else -1)
        body, shift, consumed = m.group(2), 0.0, 0
        try:
            for sign, mult, formula in _ADDUCT_PART.findall(body):
                shift += (1 if sign == '+' else -1) * (int(mult) if mult else 1) * formula_mass(formula)
                consumed += len(sign) + len(mult) + len(formula)
            if consumed == len(body):
                result = (n_mol, shift - charge * ELECTRON_MASS, charge)
        except Exception:
            result = None
    _adduct_cache[adduct] = result
    return result


def neutral_mass_from_mz(precursor_mz, adduct):
    """Monoisotopic neutral mass implied by a precursor m/z and adduct; NaN if the adduct is unknown."""
    parsed = parse_adduct(adduct)
    if parsed is None:
        return float('nan')
    n_mol, shift, charge = parsed
    return (precursor_mz * abs(charge) - shift) / n_mol


def mz_from_neutral_mass(mass, adduct):
    parsed = parse_adduct(adduct)
    if parsed is None:
        return float('nan')
    n_mol, shift, charge = parsed
    return (n_mol * mass + shift) / abs(charge)


# Measured on both timsTOF libraries in train (enveda-180, enveda-np-examples): the neutral mass implied by the
# recorded precursor m/z sits about +2.0 ppm (positive mode) / +0.6 ppm (negative mode) above the true mass.
MASS_BIAS_PPM = {1: 2.0, -1: 0.6}


def molecule_neutral_mass(precursor_mzs, adducts, calibrate=True):
    """One neutral-mass estimate for a molecule from all of its spectra (median is robust to a bad adduct)."""
    masses = []
    for mz, ad in zip(precursor_mzs, adducts):
        m = neutral_mass_from_mz(mz, ad)
        if np.isfinite(m):
            if calibrate:
                m *= 1.0 - MASS_BIAS_PPM[1 if parse_adduct(ad)[2] > 0 else -1] * 1e-6
            masses.append(m)
    return float(np.median(masses)) if masses else float('nan')


# ─────────────────────────────────────────────────────────────────────
# Structure keys — the Kaggle metric compares tautomer-canonical InChIKey first blocks
# ─────────────────────────────────────────────────────────────────────

_taut_enum = None


def _tautomer_enumerator():
    global _taut_enum
    if _taut_enum is None:
        _taut_enum = rdMolStandardize.TautomerEnumerator()
    return _taut_enum


def mol_from_smiles(smiles):
    if not smiles or not isinstance(smiles, str):
        return None
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None


def plain_key(smiles_or_mol):
    """InChIKey first block with no tautomer canonicalisation — cheap, for bulk dedup."""
    mol = mol_from_smiles(smiles_or_mol) if isinstance(smiles_or_mol, str) else smiles_or_mol
    if mol is None:
        return None
    try:
        key = Chem.MolToInchiKey(mol)
        return key.split('-')[0] if key else None
    except Exception:
        return None


def metric_key(smiles_or_mol):
    """The key the competition metric compares: tautomer-canonicalise, then InChIKey first block.

    Falls back to the plain key if canonicalisation fails, and None if the SMILES is invalid.
    """
    mol = mol_from_smiles(smiles_or_mol) if isinstance(smiles_or_mol, str) else smiles_or_mol
    if mol is None:
        return None
    try:
        canon = _tautomer_enumerator().Canonicalize(mol)
        key = Chem.MolToInchiKey(canon)
        if key:
            return key.split('-')[0]
    except Exception:
        pass
    return plain_key(mol)


def flat_canonical_smiles(smiles_or_mol):
    """Canonical SMILES with stereochemistry removed (the metric ignores it)."""
    mol = mol_from_smiles(smiles_or_mol) if isinstance(smiles_or_mol, str) else smiles_or_mol
    if mol is None:
        return None
    try:
        return Chem.MolToSmiles(mol, isomericSmiles=False)
    except Exception:
        return None


def exact_mass(smiles_or_mol):
    mol = mol_from_smiles(smiles_or_mol) if isinstance(smiles_or_mol, str) else smiles_or_mol
    if mol is None:
        return float('nan')
    # ExactMolWt ignores electrons; correct for formal charge so ions compare properly.
    return ExactMolWt(mol) - Chem.GetFormalCharge(mol) * ELECTRON_MASS


def mol_formula(smiles_or_mol):
    mol = mol_from_smiles(smiles_or_mol) if isinstance(smiles_or_mol, str) else smiles_or_mol
    return CalcMolFormula(mol) if mol is not None else None


def ppm_window(mass, ppm):
    delta = mass * ppm * 1e-6
    return mass - delta, mass + delta


# ─────────────────────────────────────────────────────────────────────
# Fingerprints
# ─────────────────────────────────────────────────────────────────────

FP_BITS = 4096
_fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_BITS)


def morgan_bits(smiles_or_mol):
    """Indices of the on-bits of the 4096-bit Morgan radius-2 fingerprint (None if invalid)."""
    mol = mol_from_smiles(smiles_or_mol) if isinstance(smiles_or_mol, str) else smiles_or_mol
    if mol is None:
        return None
    return np.fromiter(_fp_gen.GetFingerprint(mol).GetOnBits(), dtype=np.int16)


def bits_to_dense(bit_lists, n_bits=FP_BITS, dtype=np.float32):
    out = np.zeros((len(bit_lists), n_bits), dtype=dtype)
    for i, bits in enumerate(bit_lists):
        if bits is not None and len(bits):
            out[i, bits] = 1
    return out


# ─────────────────────────────────────────────────────────────────────
# Spectrum preprocessing
# ─────────────────────────────────────────────────────────────────────

MAX_PEAKS = 128


def prep_peaks(mzs, intensities, precursor_mz, max_peaks=MAX_PEAKS, min_rel_intensity=0.0):
    """Drop peaks above precursor+2 Da, keep the `max_peaks` most intense, renormalise to base peak 1.

    Returns (mzs, intensities) as float32 arrays sorted by ascending m/z.
    """
    mzs = np.asarray(mzs, dtype=np.float64)
    intensities = np.asarray(intensities, dtype=np.float64)
    keep = (mzs <= precursor_mz + 2.0) & (intensities > 0) & np.isfinite(mzs) & np.isfinite(intensities)
    mzs, intensities = mzs[keep], intensities[keep]
    if len(mzs) == 0:
        return np.zeros(0, np.float32), np.zeros(0, np.float32)
    intensities = intensities / intensities.max()
    if min_rel_intensity > 0:
        keep = intensities >= min_rel_intensity
        mzs, intensities = mzs[keep], intensities[keep]
    if len(mzs) > max_peaks:
        top = np.argpartition(intensities, -max_peaks)[-max_peaks:]
        mzs, intensities = mzs[top], intensities[top]
    order = np.argsort(mzs)
    return mzs[order].astype(np.float32), intensities[order].astype(np.float32)


def first_collision_energy(value):
    """collision_energy_ev is a list (or null); summarise as its mean, NaN when absent."""
    if value is None:
        return float('nan')
    try:
        arr = np.asarray(value, dtype=np.float64).ravel()
        arr = arr[np.isfinite(arr)]
        return float(arr.mean()) if len(arr) else float('nan')
    except Exception:
        return float('nan')


# ─────────────────────────────────────────────────────────────────────
# SMILES tokeniser (atom level — no external tokenizer dependency)
# ─────────────────────────────────────────────────────────────────────

PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3
_SPECIALS = ['<pad>', '<s>', '</s>', '<unk>']
_SMILES_TOKEN = re.compile(
    r'(\[[^\]]+\]|Br|Cl|Si|Se|se|@@|%\d{2}|[BCNOPSFIbcnops]|[()=#\-+\\/:~.*$]|\d)'
)


def split_smiles(smiles):
    return _SMILES_TOKEN.findall(smiles)


class SmilesTokenizer:
    def __init__(self, vocab):
        self.itos = list(vocab)
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}

    @classmethod
    def build(cls, smiles_iter, min_count=1):
        counts = {}
        for smi in smiles_iter:
            for tok in split_smiles(smi):
                counts[tok] = counts.get(tok, 0) + 1
        toks = sorted(t for t, c in counts.items() if c >= min_count)
        return cls(_SPECIALS + toks)

    @classmethod
    def load(cls, path):
        with open(path) as fh:
            return cls(json.load(fh))

    def save(self, path):
        with open(path, 'w') as fh:
            json.dump(self.itos, fh)

    def __len__(self):
        return len(self.itos)

    def encode(self, smiles):
        """-> [BOS, tokens..., EOS]; None if the SMILES contains an out-of-vocabulary token."""
        toks = split_smiles(smiles)
        if ''.join(toks) != smiles:
            return None
        ids = [self.stoi.get(t, UNK_ID) for t in toks]
        if UNK_ID in ids:
            return None
        return [BOS_ID] + ids + [EOS_ID]

    def decode(self, ids):
        out = []
        for i in ids:
            i = int(i)
            if i == EOS_ID:
                break
            if i in (PAD_ID, BOS_ID):
                continue
            out.append(self.itos[i] if 0 <= i < len(self.itos) else '')
        return ''.join(out)


In [ ]:
%%writefile casmi/model.py
"""
Spectrum -> structure model.

  SpectrumEncoder : peaks (m/z, neutral loss, intensity) + a metadata token (precursor m/z, neutral mass,
                    adduct, collision energy) -> contextual peak states
  FingerprintHead : pooled encoder state -> 4096-bit Morgan fingerprint logits   (ranks database candidates)
  SmilesDecoder   : autoregressive SMILES decoder with cross-attention           (de novo + candidate likelihood)

Differences from the tutorial model that matter for correctness:
  * teacher forcing shifts the labels: position i is trained to predict token i+1 (the tutorial trained
    position i to predict token i, which a causal decoder solves by copying its input);
  * generation feeds the whole prefix through a causal decoder (with a KV cache), not just the last token.
"""

import math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from casmi.common import PAD_ID, BOS_ID, EOS_ID, FP_BITS, TEST_ADDUCTS, parse_adduct

N_ADDUCTS = len(TEST_ADDUCTS) + 1   # 0 = unknown

# adduct id -> (n_mol, shift, |charge|) so the model can be told the implied neutral mass
_ADDUCT_TABLE = np.zeros((N_ADDUCTS, 3), dtype=np.float64)
_ADDUCT_TABLE[0] = (1, 0.0, 1)
for _i, _a in enumerate(TEST_ADDUCTS):
    _n, _shift, _z = parse_adduct(_a)
    _ADDUCT_TABLE[_i + 1] = (_n, _shift, abs(_z))


def default_config(vocab_size):
    return dict(vocab_size=vocab_size, d_model=512, n_heads=8, enc_layers=6, dec_layers=6, d_ff=2048,
                dropout=0.1, max_tokens=192, fp_bits=FP_BITS)


class SinusoidalMz(nn.Module):
    """Fixed sinusoidal features of a mass value, wavelengths log-spaced from 1e-2 to 1e3 Da."""

    def __init__(self, dim, min_wavelength=1e-2, max_wavelength=1e3):
        super().__init__()
        wavelengths = torch.logspace(math.log10(min_wavelength), math.log10(max_wavelength), dim // 2)
        self.register_buffer('freq', 2 * math.pi / wavelengths, persistent=False)

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            phase = x.float().unsqueeze(-1) * self.freq
            return torch.cat([phase.sin(), phase.cos()], dim=-1)


class Block(nn.Module):
    """Pre-LN transformer block: self-attention, optional cross-attention, MLP."""

    def __init__(self, d_model, n_heads, d_ff, dropout, cross=False):
        super().__init__()
        self.n_heads, self.dropout = n_heads, dropout
        self.ln1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.cross = cross
        if cross:
            self.ln_x = nn.LayerNorm(d_model)
            self.q_x = nn.Linear(d_model, d_model)
            self.kv_x = nn.Linear(d_model, 2 * d_model)
            self.proj_x = nn.Linear(d_model, d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)

    def _heads(self, x):
        b, t, d = x.shape
        return x.view(b, t, self.n_heads, d // self.n_heads).transpose(1, 2)

    def _merge(self, x):
        b, h, t, dh = x.shape
        return x.transpose(1, 2).reshape(b, t, h * dh)

    def memory_kv(self, memory):
        k, v = self.kv_x(memory).chunk(2, dim=-1)
        return self._heads(k), self._heads(v)

    def forward(self, x, attn_mask=None, causal=False, memory_kv=None, memory_mask=None, cache=None):
        """attn_mask / memory_mask: bool [B,1,1,S], True = attend. cache: dict with 'k','v' for incremental decoding."""
        p = self.dropout if self.training else 0.0
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q, k, v = self._heads(q), self._heads(k), self._heads(v)
        if cache is not None:
            if 'k' in cache:
                k = torch.cat([cache['k'], k], dim=2)
                v = torch.cat([cache['v'], v], dim=2)
            cache['k'], cache['v'] = k, v
            # a single new query attends to every cached position, so no causal mask is needed
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=p)
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=p,
                                               is_causal=causal and attn_mask is None)
        x = x + self.drop(self.proj(self._merge(y)))
        if self.cross:
            qx = self._heads(self.q_x(self.ln_x(x)))
            kx, vx = memory_kv
            y = F.scaled_dot_product_attention(qx, kx, vx, attn_mask=memory_mask, dropout_p=p)
            x = x + self.drop(self.proj_x(self._merge(y)))
        return x + self.drop(self.mlp(self.ln2(x)))


class SpectrumEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['d_model']
        self.sin = SinusoidalMz(d)
        self.peak_mlp = nn.Sequential(nn.Linear(2 * d + 2, d), nn.GELU(), nn.Linear(d, d))
        self.adduct_emb = nn.Embedding(N_ADDUCTS, 64)
        self.meta_mlp = nn.Sequential(nn.Linear(2 * d + 64 + 2, d), nn.GELU(), nn.Linear(d, d))
        self.register_buffer('adduct_table', torch.tensor(_ADDUCT_TABLE), persistent=False)
        self.blocks = nn.ModuleList([Block(d, cfg['n_heads'], cfg['d_ff'], cfg['dropout']) for _ in range(cfg['enc_layers'])])
        self.ln = nn.LayerNorm(d)
        self.drop = nn.Dropout(cfg['dropout'])

    def neutral_mass(self, prec, adduct):
        tab = self.adduct_table[adduct]
        return (prec.double() * tab[:, 2] - tab[:, 1]) / tab[:, 0]

    def forward(self, mz, inten, prec, adduct, ce):
        """mz/inten [B,P] (a slot with intensity 0 is padding), prec [B] float64, adduct [B] long, ce [B] (-1 = unknown).

        Returns (states [B,1+P,D], mask [B,1+P] bool) — position 0 is the metadata token.
        """
        b, p = mz.shape
        peak_mask = inten > 0
        loss = (prec[:, None] - mz.double()).clamp(min=0).float()
        inten = inten.float()
        peaks = self.peak_mlp(torch.cat([self.sin(mz), self.sin(loss), inten[..., None], inten.sqrt()[..., None]], dim=-1))
        has_ce = (ce >= 0).float()
        meta = self.meta_mlp(torch.cat([
            self.sin(prec.float()), self.sin(self.neutral_mass(prec, adduct).float()), self.adduct_emb(adduct),
            (ce.clamp(min=0) / 100.0)[:, None], has_ce[:, None]], dim=-1))
        x = self.drop(torch.cat([meta[:, None, :], peaks], dim=1))
        mask = torch.cat([torch.ones(b, 1, dtype=torch.bool, device=mz.device), peak_mask], dim=1)
        attn_mask = mask[:, None, None, :]
        for blk in self.blocks:
            x = blk(x, attn_mask=attn_mask)
        return self.ln(x), mask


class FingerprintHead(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['d_model']
        self.net = nn.Sequential(nn.Linear(2 * d, 2 * d), nn.GELU(), nn.Dropout(cfg['dropout']), nn.Linear(2 * d, cfg['fp_bits']))

    def forward(self, states, mask):
        m = mask[..., None].to(states.dtype)
        mean = (states * m).sum(1) / m.sum(1).clamp(min=1)
        return self.net(torch.cat([states[:, 0], mean], dim=-1))


class SmilesDecoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['d_model']
        self.max_tokens = cfg['max_tokens']
        self.tok = nn.Embedding(cfg['vocab_size'], d, padding_idx=PAD_ID)
        self.pos = nn.Embedding(cfg['max_tokens'], d)
        self.blocks = nn.ModuleList([Block(d, cfg['n_heads'], cfg['d_ff'], cfg['dropout'], cross=True) for _ in range(cfg['dec_layers'])])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, cfg['vocab_size'], bias=False)
        self.drop = nn.Dropout(cfg['dropout'])

    def forward(self, tokens_in, memory, memory_mask):
        """Teacher forcing. tokens_in [B,T] = sequence WITHOUT its last token; returns logits [B,T,V] for tokens 1..T."""
        t = tokens_in.shape[1]
        x = self.drop(self.tok(tokens_in) + self.pos(torch.arange(t, device=tokens_in.device))[None])
        mem_mask = memory_mask[:, None, None, :]
        for blk in self.blocks:
            x = blk(x, causal=True, memory_kv=blk.memory_kv(memory), memory_mask=mem_mask)
        return self.head(self.ln(x))

    def step(self, token, position, memory_kvs, mem_mask, caches):
        x = self.tok(token) + self.pos(torch.full_like(token, position))
        for blk, mkv, cache in zip(self.blocks, memory_kvs, caches):
            x = blk(x, memory_kv=mkv, memory_mask=mem_mask, cache=cache)
        return self.head(self.ln(x))[:, -1]


class CasmiModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        self.encoder = SpectrumEncoder(cfg)
        self.fp_head = FingerprintHead(cfg)
        self.decoder = SmilesDecoder(cfg)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def encode(self, batch):
        return self.encoder(batch['mz'], batch['inten'], batch['prec'], batch['adduct'], batch['ce'])

    def forward(self, batch):
        """Training forward. batch['tokens'] is [B,L] = BOS ... EOS PAD*. Returns dict of losses."""
        states, mask = self.encode(batch)
        tokens = batch['tokens']
        logits = self.decoder(tokens[:, :-1], states, mask)           # inputs : BOS t1 ... t(n-1)
        targets = tokens[:, 1:]                                        # targets: t1 ... EOS   <- the shift
        ce = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=PAD_ID)
        out = {'ce': ce}
        if 'fp' in batch:
            fp_logits = self.fp_head(states, mask).float()
            out['bce'] = F.binary_cross_entropy_with_logits(fp_logits, batch['fp'])
            out['fp_cos'] = F.cosine_similarity(fp_logits.sigmoid(), batch['fp'], dim=-1).mean()
        return out

    @torch.inference_mode()
    def predict_fingerprint(self, states, mask):
        return self.fp_head(states, mask).float().sigmoid()

    @torch.inference_mode()
    def score(self, states, mask, tokens):
        """Sum log p(tokens | spectrum) for already-paired rows: states [N,S,D], tokens [N,L] (BOS..EOS PAD*)."""
        logits = self.decoder(tokens[:, :-1], states, mask).float()
        logp = logits.log_softmax(-1).gather(-1, tokens[:, 1:, None]).squeeze(-1)
        keep = tokens[:, 1:] != PAD_ID
        return (logp * keep).sum(-1), keep.sum(-1)

    @torch.inference_mode()
    def generate(self, states, mask, n_samples=25, max_new_tokens=160, temperature=1.0, top_k=0):
        """Ancestral sampling with a KV cache. Returns (tokens [B,n,L], logprob [B,n]) — logprob is under T=1."""
        b, s, d = states.shape
        n = b * n_samples
        states = states.repeat_interleave(n_samples, dim=0)
        mem_mask = mask.repeat_interleave(n_samples, dim=0)[:, None, None, :]
        memory_kvs = [blk.memory_kv(states) for blk in self.decoder.blocks]
        caches = [dict() for _ in self.decoder.blocks]
        token = torch.full((n, 1), BOS_ID, dtype=torch.long, device=states.device)
        seqs, logprob = [token], torch.zeros(n, device=states.device)
        done = torch.zeros(n, dtype=torch.bool, device=states.device)
        max_new_tokens = min(max_new_tokens, self.decoder.max_tokens - 1)
        for position in range(max_new_tokens):
            logits = self.decoder.step(token, position, memory_kvs, mem_mask, caches).float()
            logp = logits.log_softmax(-1)
            sample_logits = logits / max(temperature, 1e-6)
            if top_k and top_k < sample_logits.size(-1):
                kth = sample_logits.topk(top_k, dim=-1).values[:, -1:]
                sample_logits = sample_logits.masked_fill(sample_logits < kth, float('-inf'))
            nxt = torch.multinomial(sample_logits.softmax(-1), 1)
            nxt = torch.where(done[:, None], torch.full_like(nxt, PAD_ID), nxt)
            logprob = logprob + torch.where(done, torch.zeros_like(logprob), logp.gather(-1, nxt).squeeze(-1))
            seqs.append(nxt)
            done = done | (nxt.squeeze(-1) == EOS_ID)
            token = nxt
            if bool(done.all()):
                break
        tokens = torch.cat(seqs, dim=1)
        logprob = torch.where(done, logprob, torch.full_like(logprob, float('-inf')))   # unfinished = invalid
        return tokens.view(b, n_samples, -1), logprob.view(b, n_samples)


def load_model(path, device='cpu'):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    model = CasmiModel(ckpt['cfg'])
    model.load_state_dict(ckpt['model'])
    return model.to(device).eval(), ckpt


In [ ]:
%%writefile casmi/infer.py
"""
Model inference helpers: raw spectra -> tensors, per-molecule fingerprint prediction, candidate
log-likelihood scoring and de novo sampling.
"""

import numpy as np
import torch

from casmi.common import (
    ADDUCT_TO_ID, MAX_PEAKS, PAD_ID, SmilesTokenizer, prep_peaks, first_collision_energy,
)
from casmi.model import load_model


def spectra_to_batch(spectra, device):
    """spectra: list of dicts with mzs, intensities, precursor_mz, adduct, collision_energy_ev (optional)."""
    n = len(spectra)
    mz = np.zeros((n, MAX_PEAKS), np.float32)
    inten = np.zeros((n, MAX_PEAKS), np.float32)
    prec = np.zeros(n, np.float64)
    adduct = np.zeros(n, np.int64)
    ce = np.full(n, -1.0, np.float32)
    for i, sp in enumerate(spectra):
        m, it = prep_peaks(sp['mzs'], sp['intensities'], sp['precursor_mz'])
        mz[i, :len(m)], inten[i, :len(m)] = m, it
        prec[i] = sp['precursor_mz']
        adduct[i] = ADDUCT_TO_ID.get(sp.get('adduct'), 0)
        c = first_collision_energy(sp.get('collision_energy_ev'))
        ce[i] = c if np.isfinite(c) else -1.0
    batch = dict(mz=mz, inten=inten, prec=prec, adduct=adduct, ce=ce)
    return {k: torch.from_numpy(v).to(device) for k, v in batch.items()}


class ModelScorer:
    def __init__(self, model_path, vocab_path, device=None):
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.model, self.ckpt = load_model(model_path, self.device)
        self.tokenizer = SmilesTokenizer.load(vocab_path)
        # bf16 only where the hardware really has it (Ampere+); a Kaggle T4 runs fp32, which is fast enough here
        self.amp = self.device.type == 'cuda' and torch.cuda.get_device_capability(self.device)[0] >= 8
        self.max_tokens = self.model.decoder.max_tokens

    def _autocast(self):
        return torch.autocast(device_type=self.device.type, dtype=torch.bfloat16, enabled=self.amp)

    @torch.inference_mode()
    def encode(self, spectra):
        with self._autocast():
            return self.model.encode(spectra_to_batch(spectra, self.device))

    @torch.inference_mode()
    def fingerprint(self, states, mask):
        """Mean predicted fingerprint probability over a molecule's spectra -> numpy [4096]."""
        with self._autocast():
            return self.model.predict_fingerprint(states, mask).mean(0).cpu().numpy()

    def tokenize(self, smiles_list):
        """-> (token matrix [N,L], ok mask). SMILES that cannot be tokenised or are too long get ok=False."""
        ids = [self.tokenizer.encode(s) if isinstance(s, str) else None for s in smiles_list]
        ok = np.array([t is not None and len(t) <= self.max_tokens for t in ids])
        width = max([len(t) for t, good in zip(ids, ok) if good], default=2)
        mat = np.full((len(ids), width), PAD_ID, np.int64)
        for i, (t, good) in enumerate(zip(ids, ok)):
            if good:
                mat[i, :len(t)] = t
        return mat, ok

    @torch.inference_mode()
    def loglik(self, states, mask, smiles_list, max_rows=768):
        """Mean over the molecule's spectra of log p(SMILES | spectrum). Returns (loglik [N], n_tokens [N]); -inf if unscorable."""
        n_spec = states.shape[0]
        mat, ok = self.tokenize(smiles_list)
        out = np.full(len(smiles_list), -np.inf, np.float64)
        n_tok = (mat != PAD_ID).sum(1) - 1
        good = np.flatnonzero(ok)
        order = good[np.argsort(n_tok[good])]                  # similar lengths together -> little padding
        chunk = max(1, max_rows // n_spec)
        for i in range(0, len(order), chunk):
            sel = order[i:i + chunk]
            width = int(n_tok[sel].max()) + 1
            toks = torch.from_numpy(mat[sel, :width]).to(self.device)
            c = len(sel)
            st = states.repeat(c, 1, 1)                        # [c*S, P, D] : candidate-major blocks of S spectra
            mk = mask.repeat(c, 1)
            tk = toks.repeat_interleave(n_spec, dim=0)
            with self._autocast():
                lp, _ = self.model.score(st, mk, tk)
            out[sel] = lp.view(c, n_spec).mean(1).double().cpu().numpy()
        return out, n_tok

    @torch.inference_mode()
    def sample(self, states, mask, n_samples=64, temperature=1.0, top_k=0, max_new_tokens=160, max_rows=1024):
        """De novo sampling for every spectrum. Returns list of (smiles, logprob) over all spectra (finished samples only)."""
        results = []
        per_call = max(1, max_rows // n_samples)
        for i in range(0, states.shape[0], per_call):
            with self._autocast():
                tokens, logprob = self.model.generate(states[i:i + per_call], mask[i:i + per_call], n_samples=n_samples,
                                                      temperature=temperature, top_k=top_k, max_new_tokens=max_new_tokens)
            tokens, logprob = tokens.cpu().numpy(), logprob.float().cpu().numpy()
            for b in range(tokens.shape[0]):
                for j in range(tokens.shape[1]):
                    if np.isfinite(logprob[b, j]):
                        results.append((self.tokenizer.decode(tokens[b, j, 1:]), float(logprob[b, j])))
        return results


In [ ]:
%%writefile casmi/libsearch.py
"""
Spectral library search for the CASMI 2026 pipeline (class-1 molecules).

For a molecule we only have its MS/MS spectra.  ``estimate_molecule_mass`` turns the
precursor m/z values into a calibrated neutral mass; ``SpectralLibrary`` holds every
training spectrum whose *structure* mass falls into one of the query mass windows, and
``SpectralLibrary.search`` ranks the structures in one window by how well their library
spectra match the query spectra.

numpy + pandas + pyarrow only -- no numba, no rdkit at search time, so the same file runs
unchanged in a Kaggle notebook.

    from casmi.libsearch import SpectralLibrary, SearchConfig, estimate_molecule_mass

    mass = estimate_molecule_mass(prec_mzs, adducts, modes)
    lo, hi = mass * (1 - 10e-6), mass * (1 + 10e-6)
    lib = SpectralLibrary.from_train('train.parquet', structures_df, [(lo, hi)])
    hits = lib.search(query_spectra, lo, hi)
"""

import os
import time
import numpy as np
import pandas as pd

try:                                    # pyarrow is only needed to build the library
    import pyarrow.parquet as pq
except Exception:                       # pragma: no cover
    pq = None


# ─────────────────────────────────────────────────────────────────────
# Adduct arithmetic (duplicated from casmi.common so this module has no rdkit import)
# ─────────────────────────────────────────────────────────────────────

ELECTRON_MASS = 0.00054857990907

# (n_mol, mass_shift, charge); m/z = (n_mol * M + shift) / |charge|
_ADDUCT_TABLE = {
    '[M+H]+':        (1,   1.00727645, 1),
    '[M+NH4]+':      (1,  18.03382555, 1),
    '[M-H2O+H]+':    (1, -17.00328247, 1),
    '[M-2H2O+H]+':   (1, -35.01384139, 1),
    '[M+Na]+':       (1,  22.98922123, 1),
    '[M+K]+':        (1,  38.96315773, 1),
    '[M-H]-':        (1,  -1.00727645, -1),
    '[M-H2O-H]-':    (1, -19.01784037, -1),
    '[M+CH2O2-H]-':  (1,  44.99820285, -1),
    '[M+Cl]-':       (1,  34.96940140, -1),
}

# Everything else that occurs in the training libraries is parsed from the string.
_PT_MASS = {
    'H': 1.00782503207, 'D': 2.0141017778, 'C': 12.0, 'N': 14.0030740048, 'O': 15.9949146196,
    'F': 18.99840322, 'Na': 22.9897692809, 'Mg': 23.985041700, 'Si': 27.9769265325,
    'P': 30.97376163, 'S': 31.97207100, 'Cl': 34.96885268, 'K': 38.96370668,
    'Ca': 39.96259098, 'Fe': 55.9349375, 'Ni': 57.9353429, 'Cu': 62.9295975,
    'Zn': 63.9291422, 'Br': 78.9183371, 'I': 126.904473, 'Li': 7.01600455,
    'Ac': 42.01056468,          # acetyl/acetic-acid shorthand used by some libraries
    'FA': 46.00547931, 'Hac': 60.02112937, 'TFA': 113.99286,
}

_adduct_cache = dict(_ADDUCT_TABLE)


def _formula_mass(formula):
    import re
    total, consumed = 0.0, 0
    for sym, count in re.findall(r'([A-Z][a-z]?)(\d*)', formula):
        if not sym:
            continue
        if sym not in _PT_MASS:
            return None
        total += _PT_MASS[sym] * (int(count) if count else 1)
        consumed += len(sym) + len(count)
    return total if consumed == len(formula) else None


def parse_adduct(adduct):
    """'[2M+Na-2H]-' -> (n_mol, mass_shift, charge); None when unparsable."""
    if adduct in _adduct_cache:
        return _adduct_cache[adduct]
    import re
    result = None
    m = re.match(r'^\[(\d*)M([^\]]*)\](\d*)([+-])$', adduct.replace(' ', '')) if isinstance(adduct, str) else None
    if m:
        n_mol = int(m.group(1)) if m.group(1) else 1
        charge = (int(m.group(3)) if m.group(3) else 1) * (1 if m.group(4) == '+' else -1)
        body, shift, consumed, ok = m.group(2), 0.0, 0, True
        for sign, mult, formula in re.findall(r'([+-])(\d*)([A-Za-z][A-Za-z0-9]*)', body):
            fm = _formula_mass(formula)
            if fm is None:
                ok = False
                break
            shift += (1 if sign == '+' else -1) * (int(mult) if mult else 1) * fm
            consumed += len(sign) + len(mult) + len(formula)
        if ok and consumed == len(body):
            result = (n_mol, shift - charge * ELECTRON_MASS, charge)
    _adduct_cache[adduct] = result
    return result


def neutral_mass_from_mz(precursor_mz, adduct):
    parsed = parse_adduct(adduct)
    if parsed is None:
        return float('nan')
    n_mol, shift, charge = parsed
    return (precursor_mz * abs(charge) - shift) / n_mol


# ─────────────────────────────────────────────────────────────────────
# Mass calibration
# ─────────────────────────────────────────────────────────────────────

# Measured-minus-theoretical neutral mass on the Bruker timsTOF used for the hidden test
# set.  On enveda-np-examples alone (same instrument and pipeline as the test set) the
# median is +1.89 ppm positive / +0.45 ppm negative; casmi.common uses +2.0 / +0.6 measured
# over both timsTOF libraries.  These values are only a fallback -- see estimate_molecule_mass.
PPM_BIAS = {'positive': 2.0, 'negative': 0.6}
DEFAULT_MASS_PPM = 10.0


def _mode_key(mode):
    if mode is None:
        return 'positive'
    m = str(mode).strip().lower()
    if m.startswith('n') or m in ('-', 'neg'):
        return 'negative'
    return 'positive'


def estimate_molecule_mass(precursor_mzs, adducts, ion_modes=None):
    """Calibrated neutral mass of one molecule from all of its spectra (median, robust).

    Delegates to ``casmi.common.molecule_neutral_mass`` (the shared, jointly-calibrated
    implementation) when it is importable; the fallback below repeats the same arithmetic
    so this module also works on its own.  NaN when no spectrum has a parsable adduct.
    """
    try:
        from casmi.common import molecule_neutral_mass
        return molecule_neutral_mass(list(precursor_mzs), list(adducts), calibrate=True)
    except Exception:
        pass
    modes = ion_modes if ion_modes is not None else [None] * len(list(precursor_mzs))
    out = []
    for mz, ad, mode in zip(precursor_mzs, adducts, modes):
        m = neutral_mass_from_mz(float(mz), ad)
        if not (np.isfinite(m) and m > 0):
            continue
        if mode is None:
            parsed = parse_adduct(ad)
            mode = 'positive' if (parsed and parsed[2] > 0) else 'negative'
        out.append(m * (1.0 - PPM_BIAS[_mode_key(mode)] * 1e-6))
    return float(np.median(out)) if out else float('nan')


def mass_window(mass, ppm=DEFAULT_MASS_PPM):
    d = mass * ppm * 1e-6
    return mass - d, mass + d


# ─────────────────────────────────────────────────────────────────────
# Search configuration
# ─────────────────────────────────────────────────────────────────────

class SearchConfig(object):
    """Everything that can be tuned about the similarity.  Cheap to change (see `set_config`).

    The defaults are the configuration chosen by the validation sweep in casmi.libsearch_eval
    (see work/libsearch/REPORT.md): sqrt-intensity cosine with greedy one-to-one matching at
    0.02 Da, precursor region removed, a 0.5% relative-intensity floor, the 60 most intense
    peaks, and per-molecule aggregation by the mean of the two best query spectra.
    """

    def __init__(self,
                 tol_da=0.02,               # peak-matching tolerance in Da
                 tol_ppm=0.0,               # extra m/z-proportional tolerance (added to tol_da)
                 power=0.5,                 # intensity transform i**power (1.0 = none, 0.5 = sqrt)
                 mz_power=0.0,              # optional m/z weighting: i**power * mz**mz_power
                 min_rel=0.005,             # relative-intensity floor applied to both sides
                 max_peaks=60,              # top-N peaks kept per spectrum
                 drop_precursor=1.5,        # remove peaks within this many Da of the precursor (0 = keep)
                 modified=False,            # also match neutral-loss-shifted peaks (modified cosine)
                 min_matches=2,             # cosine forced to 0 below this many matched peaks
                 same_mode_only=True,       # only compare spectra of the same ionization mode
                 same_adduct_bonus=0.0,     # multiply cross-adduct cosines by (1 - this)
                 agg='mean_top2',           # 'max' | 'mean_qmax' | 'mean_qmax_all' | 'mean_topK'
                 ):
        self.tol_da = float(tol_da)
        self.tol_ppm = float(tol_ppm)
        self.power = float(power)
        self.mz_power = float(mz_power)
        self.min_rel = float(min_rel)
        self.max_peaks = int(max_peaks)
        self.drop_precursor = float(drop_precursor)
        self.modified = bool(modified)
        self.min_matches = int(min_matches)
        self.same_mode_only = bool(same_mode_only)
        self.same_adduct_bonus = float(same_adduct_bonus)
        self.agg = str(agg)

    def copy(self, **kw):
        c = SearchConfig(**self.__dict__)
        for k, v in kw.items():
            setattr(c, k, v)
        return c

    def __repr__(self):
        d = self.__dict__
        return 'SearchConfig(' + ', '.join('%s=%r' % (k, d[k]) for k in sorted(d)) + ')'


DEFAULT_CONFIG = SearchConfig()


# ─────────────────────────────────────────────────────────────────────
# Peak preprocessing
# ─────────────────────────────────────────────────────────────────────

def prep_spectrum(mzs, intensities, precursor_mz, max_peaks=256, min_rel=0.0, drop_precursor=0.0):
    """Clean one spectrum -> (mz, intensity) float32 arrays sorted by m/z, base peak = 1."""
    mz = np.asarray(mzs, dtype=np.float64).ravel()
    it = np.asarray(intensities, dtype=np.float64).ravel()
    n = min(len(mz), len(it))
    mz, it = mz[:n], it[:n]
    keep = np.isfinite(mz) & np.isfinite(it) & (it > 0) & (mz > 0)
    if precursor_mz and np.isfinite(precursor_mz):
        keep &= mz <= precursor_mz + 2.0
        if drop_precursor > 0:
            keep &= np.abs(mz - precursor_mz) > drop_precursor
    mz, it = mz[keep], it[keep]
    if len(mz) == 0:
        return np.zeros(0, np.float32), np.zeros(0, np.float32)
    it = it / it.max()
    if min_rel > 0:
        keep = it >= min_rel
        mz, it = mz[keep], it[keep]
        if len(mz) == 0:
            return np.zeros(0, np.float32), np.zeros(0, np.float32)
    if len(mz) > max_peaks:
        top = np.argpartition(it, -max_peaks)[-max_peaks:]
        mz, it = mz[top], it[top]
    order = np.argsort(mz, kind='stable')
    return mz[order].astype(np.float32), it[order].astype(np.float32)


def _transform(mz, it, cfg):
    """Apply the config's intensity transform; returns float64 weights."""
    w = np.asarray(it, dtype=np.float64)
    if cfg.power != 1.0:
        w = w ** cfg.power
    if cfg.mz_power != 0.0:
        w = w * (np.asarray(mz, dtype=np.float64) ** cfg.mz_power)
    return w


# ─────────────────────────────────────────────────────────────────────
# Core similarity: one query spectrum vs. a block of library spectra
# ─────────────────────────────────────────────────────────────────────

def _ranges(starts, counts):
    """Concatenated index ranges [starts[i], starts[i]+counts[i]) without a python loop."""
    total = int(counts.sum())
    if total == 0:
        return np.zeros(0, np.int64)
    counts = counts.astype(np.int64)
    out = np.arange(total, dtype=np.int64) - np.repeat(np.cumsum(counts) - counts, counts)
    return out + np.repeat(starts.astype(np.int64), counts)


def _segment_max(values, starts, counts):
    """Per-segment maximum of `values` along the last axis; empty segments give 0."""
    out = np.zeros(values.shape[:-1] + (len(starts),), dtype=values.dtype)
    ok = counts > 0
    if ok.any():
        red = np.maximum.reduceat(values, starts[ok], axis=-1)
        out[..., ok] = red
    return out


def _segment_any(values, starts, counts):
    out = np.zeros(values.shape[:-1] + (len(starts),), dtype=bool)
    ok = counts > 0
    if ok.any():
        red = np.logical_or.reduceat(values, starts[ok], axis=-1)
        out[..., ok] = red
    return out


def _match_pairs(qmz, gmz, tol_da, tol_ppm):
    """Candidate (query peak, library peak) index pairs with |dmz| <= tol.

    `gmz` must be sorted ascending.  Fully vectorised: one searchsorted per side.
    """
    tol = tol_da + qmz * tol_ppm * 1e-6
    lo = np.searchsorted(gmz, qmz - tol, side='left')
    hi = np.searchsorted(gmz, qmz + tol, side='right')
    cnt = hi - lo
    total = int(cnt.sum())
    if total == 0:
        return np.zeros(0, np.int64), np.zeros(0, np.int64)
    qi = np.repeat(np.arange(len(qmz), dtype=np.int64), cnt)
    # ranges [lo_i, hi_i) concatenated without a python loop
    starts = np.repeat(lo.astype(np.int64), cnt)
    offs = np.arange(total, dtype=np.int64) - np.repeat(np.cumsum(cnt) - cnt, cnt)
    return qi, starts + offs


def _greedy_one_to_one(qi, gj, prod, spec_of_peak, n_qpeaks):
    """Approximate greedy one-to-one matching, vectorised.

    Pairs are visited in descending intensity product; each (library spectrum, query peak)
    slot and each library peak may be used at most once.  This reproduces exact greedy
    matching whenever the winning pair of a peak is unambiguous, which is the normal case
    at the tolerances used here.
    """
    if len(qi) == 0:
        return qi, gj, prod
    order = np.argsort(-prod, kind='stable')
    qi, gj, prod = qi[order], gj[order], prod[order]
    # each query peak may be used once per library spectrum
    key = spec_of_peak[gj] * np.int64(n_qpeaks) + qi
    _, first = np.unique(key, return_index=True)
    first.sort()
    qi, gj, prod = qi[first], gj[first], prod[first]
    # each library peak may be used once (peak ids are globally unique)
    _, first = np.unique(gj, return_index=True)
    first.sort()
    return qi[first], gj[first], prod[first]


class _PeakBlock(object):
    """All peaks of a set of library spectra, concatenated and globally sorted by m/z."""

    def __init__(self, mz, weight, spec_idx):
        order = np.argsort(mz, kind='stable')
        self.mz = np.ascontiguousarray(mz[order], dtype=np.float64)
        self.weight = np.ascontiguousarray(weight[order], dtype=np.float64)
        self.spec = np.ascontiguousarray(spec_idx[order], dtype=np.int64)


def _cosine_block(qmz, qw, block, n_spec, cfg, shifts=None):
    """Cosine numerators + matched-peak counts of one query spectrum against `n_spec` library spectra.

    `shifts` (optional, length n_spec) gives a per-library-spectrum precursor difference for
    modified-cosine matching.  Returns (numerator[n_spec], n_matched[n_spec]).
    """
    num = np.zeros(n_spec, dtype=np.float64)
    cnt = np.zeros(n_spec, dtype=np.float64)
    if len(qmz) == 0 or len(block.mz) == 0:
        return num, cnt

    qi, gj = _match_pairs(qmz, block.mz, cfg.tol_da, cfg.tol_ppm)

    if shifts is not None:
        # neutral-loss channel: query peak at qmz matches a library peak at qmz - shift
        uniq = np.unique(shifts)
        uniq = uniq[np.abs(uniq) > cfg.tol_da]
        for sh in uniq:
            qi2, gj2 = _match_pairs(qmz - sh, block.mz, cfg.tol_da, cfg.tol_ppm)
            if len(qi2) == 0:
                continue
            ok = shifts[block.spec[gj2]] == sh
            if ok.any():
                qi = np.concatenate([qi, qi2[ok]])
                gj = np.concatenate([gj, gj2[ok]])
        if len(qi):
            key = block.spec[gj] * np.int64(len(qmz) + 1) + qi
            dedup = np.unique(key, return_index=True)[1]
            qi, gj = qi[dedup], gj[dedup]

    if len(qi) == 0:
        return num, cnt
    prod = qw[qi] * block.weight[gj]
    qi, gj, prod = _greedy_one_to_one(qi, gj, prod, block.spec, len(qmz))
    s = block.spec[gj]
    num += np.bincount(s, weights=prod, minlength=n_spec)[:n_spec]
    cnt += np.bincount(s, minlength=n_spec)[:n_spec]
    return num, cnt


# ─────────────────────────────────────────────────────────────────────
# The library
# ─────────────────────────────────────────────────────────────────────

_CHEAP_COLS = ['ingest_lib', 'inchikey14', 'adduct', 'ionization_mode',
               'precursor_mz', 'precursor_error_ppm']
_PEAK_COLS = ['ms2_mzs', 'ms2_normalized_intensities']

RAW_MAX_PEAKS = 256          # how many peaks are stored per library spectrum
RAW_MIN_REL = 0.002          # storage-time intensity floor (well below any config's floor)


class SpectralLibrary(object):
    """Training spectra restricted to a set of neutral-mass windows, ready for searching."""

    def __init__(self, ik14, smiles, struct_mass, n_struct_spectra,
                 spec_struct, spec_prec, spec_mode, spec_adduct, spec_lib,
                 peak_mz, peak_int, peak_off, config=None, build_seconds=0.0):
        self.ik14 = np.asarray(ik14, dtype=object)              # [n_struct]
        self.smiles = np.asarray(smiles, dtype=object)
        self.struct_mass = np.asarray(struct_mass, dtype=np.float64)
        self.n_struct_spectra = np.asarray(n_struct_spectra, dtype=np.int64)
        self.spec_struct = np.asarray(spec_struct, dtype=np.int32)   # [n_spec] -> struct row
        self.spec_prec = np.asarray(spec_prec, dtype=np.float64)
        self.spec_mode = np.asarray(spec_mode, dtype=np.int8)        # +1 positive, -1 negative
        self.spec_adduct = np.asarray(spec_adduct, dtype=object)
        self.spec_lib = np.asarray(spec_lib, dtype=object)
        self.peak_mz = np.asarray(peak_mz, dtype=np.float32)
        self.peak_int = np.asarray(peak_int, dtype=np.float32)
        self.peak_off = np.asarray(peak_off, dtype=np.int64)         # [n_spec + 1]
        self.build_seconds = float(build_seconds)
        self._mass_order = np.argsort(self.struct_mass, kind='stable')
        self._sorted_mass = self.struct_mass[self._mass_order]
        self._spec_order = np.argsort(self.spec_struct, kind='stable')
        ss = self.spec_struct[self._spec_order]
        self._spec_start = np.searchsorted(ss, np.arange(len(self.ik14) + 1))
        self.config = None
        self.set_config(config or DEFAULT_CONFIG)

    # ── construction ────────────────────────────────────────────────

    @classmethod
    def from_train(cls, train_path, structures_df, mass_windows, exclude_libs=(),
                   max_ppm_error=20.0, include_libs=None, config=None,
                   max_peaks=RAW_MAX_PEAKS, min_rel=RAW_MIN_REL, verbose=True,
                   row_groups=None):
        """Stream train.parquet and keep the spectra of structures inside `mass_windows`.

        structures_df: DataFrame with at least ik14, smiles, mass (and optionally n_spectra).
        mass_windows : list of (lo, hi) neutral-mass intervals; overlapping ones are merged.
        exclude_libs : ingest_lib values to drop (e.g. the validation library itself).
        max_ppm_error: rows with |precursor_error_ppm| above this (or NaN) are dropped --
                       they have unreliable adduct/structure labels.
        """
        if pq is None:
            raise RuntimeError('pyarrow is required to build a SpectralLibrary')
        t0 = time.time()
        exclude_libs = set(exclude_libs or ())
        include_libs = set(include_libs) if include_libs is not None else None

        lo, hi = _merge_windows(mass_windows)
        s_mass = structures_df['mass'].to_numpy(dtype=np.float64)
        inside = _in_windows(s_mass, lo, hi)
        sel = structures_df.loc[inside].reset_index(drop=True)
        if verbose:
            print('[libsearch] %d structures inside %d mass windows' % (len(sel), len(lo)), flush=True)

        ik_index = {k: i for i, k in enumerate(sel['ik14'].tolist())}
        n_struct_spectra = (sel['n_spectra'].to_numpy(dtype=np.int64)
                            if 'n_spectra' in sel.columns else np.zeros(len(sel), np.int64))

        spec_struct, spec_prec, spec_mode, spec_adduct, spec_lib = [], [], [], [], []
        mz_parts, int_parts, lens = [], [], []

        pf = pq.ParquetFile(train_path)
        rgs = range(pf.metadata.num_row_groups) if row_groups is None else list(row_groups)
        for rg in rgs:
            meta = pf.read_row_group(rg, columns=_CHEAP_COLS).to_pandas()
            keep = np.asarray(meta['inchikey14'].isin(ik_index).to_numpy(), dtype=bool).copy()
            if exclude_libs:
                keep &= ~meta['ingest_lib'].isin(exclude_libs).to_numpy()
            if include_libs is not None:
                keep &= meta['ingest_lib'].isin(include_libs).to_numpy()
            err = meta['precursor_error_ppm'].to_numpy(dtype=np.float64)
            keep &= np.isfinite(err) & (np.abs(err) <= max_ppm_error)
            idx = np.flatnonzero(keep)
            if verbose:
                print('[libsearch] row group %2d: %6d / %d kept' % (rg, len(idx), len(meta)), flush=True)
            if len(idx) == 0:
                continue
            meta = meta.iloc[idx]
            tab = pf.read_row_group(rg, columns=_PEAK_COLS).take(idx)
            mzs_list = tab.column(0).combine_chunks()
            int_list = tab.column(1).combine_chunks()
            m_off = np.asarray(mzs_list.offsets)
            m_val = np.asarray(mzs_list.values)
            i_val = np.asarray(int_list.values)
            i_off = np.asarray(int_list.offsets)
            del tab

            precs = meta['precursor_mz'].to_numpy(dtype=np.float64)
            struct_ids = [ik_index[k] for k in meta['inchikey14']]
            modes = [1 if _mode_key(m) == 'positive' else -1 for m in meta['ionization_mode']]
            adducts = meta['adduct'].tolist()
            libs = meta['ingest_lib'].tolist()

            for k in range(len(precs)):
                mz, it = prep_spectrum(m_val[m_off[k]:m_off[k + 1]], i_val[i_off[k]:i_off[k + 1]],
                                       precs[k], max_peaks=max_peaks, min_rel=min_rel)
                if len(mz) < 2:
                    continue
                mz_parts.append(mz)
                int_parts.append(it)
                lens.append(len(mz))
                spec_struct.append(struct_ids[k])
                spec_prec.append(precs[k])
                spec_mode.append(modes[k])
                spec_adduct.append(adducts[k])
                spec_lib.append(libs[k])
            del m_val, i_val, meta

        if lens:
            peak_mz = np.concatenate(mz_parts)
            peak_int = np.concatenate(int_parts)
            peak_off = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)
        else:
            peak_mz = np.zeros(0, np.float32)
            peak_int = np.zeros(0, np.float32)
            peak_off = np.zeros(1, np.int64)
        dt = time.time() - t0
        if verbose:
            print('[libsearch] kept %d spectra / %d peaks in %.1f s' % (len(lens), len(peak_mz), dt), flush=True)
        return cls(sel['ik14'].to_numpy(), sel['smiles'].to_numpy(), s_mass[inside], n_struct_spectra,
                   spec_struct, spec_prec, spec_mode, spec_adduct, spec_lib,
                   peak_mz, peak_int, peak_off, config=config, build_seconds=dt)

    # ── persistence (validation convenience; not needed on Kaggle) ──

    def save(self, path):
        np.savez(path, ik14=self.ik14, smiles=self.smiles, struct_mass=self.struct_mass,
                 n_struct_spectra=self.n_struct_spectra, spec_struct=self.spec_struct,
                 spec_prec=self.spec_prec, spec_mode=self.spec_mode, spec_adduct=self.spec_adduct,
                 spec_lib=self.spec_lib, peak_mz=self.peak_mz, peak_int=self.peak_int,
                 peak_off=self.peak_off, build_seconds=self.build_seconds)
        return path

    @classmethod
    def load(cls, path, config=None):
        d = np.load(path, allow_pickle=True)
        return cls(d['ik14'], d['smiles'], d['struct_mass'], d['n_struct_spectra'],
                   d['spec_struct'], d['spec_prec'], d['spec_mode'], d['spec_adduct'],
                   d['spec_lib'], d['peak_mz'], d['peak_int'], d['peak_off'],
                   config=config, build_seconds=float(d['build_seconds']))

    def drop_libs(self, libs):
        """A copy of this library without the given ingest_lib values (validation helper)."""
        keep = ~np.isin(self.spec_lib, list(libs))
        idx = np.flatnonzero(keep)
        counts = self.peak_off[idx + 1] - self.peak_off[idx]
        gather = _ranges(self.peak_off[idx], counts)
        off = np.zeros(len(idx) + 1, np.int64)
        np.cumsum(counts, out=off[1:])
        return SpectralLibrary(self.ik14, self.smiles, self.struct_mass, self.n_struct_spectra,
                               self.spec_struct[idx], self.spec_prec[idx], self.spec_mode[idx],
                               self.spec_adduct[idx], self.spec_lib[idx],
                               self.peak_mz[gather], self.peak_int[gather], off,
                               config=self.config, build_seconds=self.build_seconds)

    # ── config ──────────────────────────────────────────────────────

    def set_config(self, config):
        """(Re)apply the intensity transform / peak filters.  Fully vectorised, no file access."""
        self.config = config
        cfg = config
        n = len(self.spec_prec)
        off = self.peak_off
        n_peaks = len(self.peak_mz)
        if n == 0 or n_peaks == 0:
            self._cmz = np.zeros(0)
            self._cw = np.zeros(0)
            self._coff = np.zeros(n + 1, np.int64)
            self._cnorm = np.zeros(n)
            return self

        counts0 = off[1:] - off[:-1]
        owner = np.repeat(np.arange(n, dtype=np.int64), counts0)
        mz = self.peak_mz.astype(np.float64)
        it = self.peak_int.astype(np.float64)

        keep = np.ones(n_peaks, bool)
        if cfg.drop_precursor > 0:
            keep &= np.abs(mz - self.spec_prec[owner]) > cfg.drop_precursor
        if cfg.min_rel > 0:
            # renormalise to the base peak of what survived, then apply the relative floor
            masked = np.where(keep, it, 0.0)
            mx = _segment_max(masked, off[:-1], counts0)
            keep &= it >= cfg.min_rel * np.maximum(mx[owner], 1e-30)
        if cfg.max_peaks > 0:
            cnt = np.bincount(owner[keep], minlength=n) if keep.any() else np.zeros(n, np.int64)
            if (cnt > cfg.max_peaks).any():
                idx = np.flatnonzero(keep)
                order = np.lexsort((-it[idx], owner[idx]))     # by spectrum, intensity desc
                s_owner = owner[idx][order]
                starts = np.searchsorted(s_owner, np.arange(n))
                rank = np.arange(len(order), dtype=np.int64) - starts[s_owner]
                drop = idx[order[rank >= cfg.max_peaks]]
                keep[drop] = False

        mz, it, owner = mz[keep], it[keep], owner[keep]
        lens = np.bincount(owner, minlength=n)
        coff = np.zeros(n + 1, np.int64)
        np.cumsum(lens, out=coff[1:])
        # peaks are already in ascending-m/z order inside each spectrum and `keep` preserves it
        self._cmz = mz
        self._cw = _transform(mz, it, cfg)
        self._coff = coff
        sq = self._cw * self._cw
        self._cnorm = np.sqrt(np.bincount(owner, weights=sq, minlength=n))
        return self

    # ── search ──────────────────────────────────────────────────────

    def candidates(self, mass_lo, mass_hi):
        """Row indices (into self.ik14) of the structures inside the mass window."""
        a = np.searchsorted(self._sorted_mass, mass_lo, side='left')
        b = np.searchsorted(self._sorted_mass, mass_hi, side='right')
        return np.sort(self._mass_order[a:b])

    def _spectra_of(self, struct_rows):
        starts = self._spec_start[struct_rows]
        counts = self._spec_start[struct_rows + 1] - starts
        return self._spec_order[_ranges(starts, counts)]

    def search(self, query_spectra, mass_lo, mass_hi, exclude_libs=(), exclude_ik14=()):
        """Rank the structures in [mass_lo, mass_hi] by spectral similarity to `query_spectra`.

        query_spectra: list of dicts with keys mzs, intensities, precursor_mz, adduct,
                       ionization_mode -- ALL spectra of one molecule.
        exclude_libs : ingest_lib names whose spectra are ignored for THIS query only
                       (e.g. the library the validation query itself came from).
        exclude_ik14 : structures whose spectra are ignored for THIS query only -- they still
                       appear as candidates with score 0.  Used to simulate a class-2/3
                       molecule whose true structure has no reference spectra.
        Both exclusions are applied per call, so one library can serve every query.

        Returns a DataFrame (one row per candidate structure, best first) with columns
        ik14, smiles, mass, score, n_lib_spectra, best_cosine, mean_qmax, n_matched_best,
        matched_mode, same_adduct, n_train_spectra.
        """
        cfg = self.config
        rows = self.candidates(mass_lo, mass_hi)
        n_cand = len(rows)
        base = pd.DataFrame({
            'ik14': self.ik14[rows],
            'smiles': self.smiles[rows],
            'mass': self.struct_mass[rows],
            'n_train_spectra': self.n_struct_spectra[rows],
        })
        if n_cand == 0:
            for c, dt in (('score', np.float64), ('n_lib_spectra', np.int64), ('best_cosine', np.float64),
                          ('mean_qmax', np.float64), ('n_matched_best', np.int32),
                          ('matched_mode', bool), ('same_adduct', bool)):
                base[c] = np.zeros(0, dt)
            return base

        spec_ids = self._spectra_of(rows)
        # candidate-structure row -> position in `rows`
        pos_of_struct = np.zeros(len(self.ik14), np.int64)
        pos_of_struct[rows] = np.arange(n_cand)
        if len(exclude_ik14):
            blind = np.isin(self.ik14[rows], np.atleast_1d(exclude_ik14))
            if blind.any():
                spec_ids = spec_ids[~blind[pos_of_struct[self.spec_struct[spec_ids]]]]
        if len(exclude_libs):
            spec_ids = spec_ids[~np.isin(self.spec_lib[spec_ids], np.atleast_1d(exclude_libs))]
        n_lib = len(spec_ids)
        lib_cand = pos_of_struct[self.spec_struct[spec_ids]]
        n_lib_spectra = np.bincount(lib_cand, minlength=n_cand)

        if n_lib == 0 or not query_spectra:
            base['score'] = 0.0
            base['n_lib_spectra'] = n_lib_spectra
            base['best_cosine'] = 0.0
            base['mean_qmax'] = 0.0
            base['n_matched_best'] = 0
            base['matched_mode'] = False
            base['same_adduct'] = False
            return _finalise(base)

        # concatenated, globally m/z-sorted peak block for the whole candidate set
        local = np.arange(n_lib, dtype=np.int64)
        counts = self._coff[spec_ids + 1] - self._coff[spec_ids]
        gather = _ranges(self._coff[spec_ids], counts)
        block = _PeakBlock(self._cmz[gather], self._cw[gather], np.repeat(local, counts))
        lnorm = self._cnorm[spec_ids]
        lmode = self.spec_mode[spec_ids]
        lprec = self.spec_prec[spec_ids]
        ladd = self.spec_adduct[spec_ids]

        n_q = len(query_spectra)
        cos = np.zeros((n_q, n_lib), dtype=np.float32)
        nmatch = np.zeros((n_q, n_lib), dtype=np.int32)
        q_mode_ok = np.zeros((n_q, n_lib), dtype=bool)
        q_add_ok = np.zeros((n_q, n_lib), dtype=bool)

        for qi_, q in enumerate(query_spectra):
            prec = float(q.get('precursor_mz', np.nan))
            qmz, qit = prep_spectrum(q['mzs'], q['intensities'], prec,
                                     max_peaks=cfg.max_peaks, min_rel=cfg.min_rel,
                                     drop_precursor=cfg.drop_precursor)
            if len(qmz) < 2:
                continue
            qw = _transform(qmz, qit, cfg)
            qnorm = np.sqrt((qw * qw).sum())
            if qnorm <= 0:
                continue
            qmode = 1 if _mode_key(q.get('ionization_mode')) == 'positive' else -1
            mode_ok = lmode == qmode
            add_ok = ladd == q.get('adduct')
            q_mode_ok[qi_] = mode_ok
            q_add_ok[qi_] = add_ok
            use = mode_ok if cfg.same_mode_only else np.ones(n_lib, bool)
            if not use.any():
                continue
            if use.all():
                sub_block, sub_norm, sub_map = block, lnorm, None
            else:
                pkeep = use[block.spec]
                remap = np.full(n_lib, -1, np.int64)
                sub_ids = np.flatnonzero(use)
                remap[sub_ids] = np.arange(len(sub_ids))
                sub_block = _PeakBlock(block.mz[pkeep], block.weight[pkeep], remap[block.spec[pkeep]])
                sub_norm = lnorm[sub_ids]
                sub_map = sub_ids
            n_sub = n_lib if sub_map is None else len(sub_map)
            shifts = None
            if cfg.modified and np.isfinite(prec):
                sp = (prec - (lprec if sub_map is None else lprec[sub_map]))
                shifts = np.round(sp, 4)
            num, cnt = _cosine_block(qmz, qw, sub_block, n_sub, cfg, shifts=shifts)
            c = num / (qnorm * np.maximum(sub_norm, 1e-12))
            if cfg.min_matches > 1:
                c = np.where(cnt >= cfg.min_matches, c, 0.0)
            if cfg.same_adduct_bonus > 0:
                sa = add_ok if sub_map is None else add_ok[sub_map]
                c = c * np.where(sa, 1.0, 1.0 - cfg.same_adduct_bonus)
            c = np.clip(c, 0.0, 1.0)
            if sub_map is None:
                cos[qi_] = c
                nmatch[qi_] = cnt
            else:
                cos[qi_, sub_map] = c
                nmatch[qi_, sub_map] = cnt

        return _aggregate(base, cos, nmatch, lib_cand, n_cand, n_lib_spectra,
                          q_mode_ok, q_add_ok, cfg)


def _aggregate(base, cos, nmatch, lib_cand, n_cand, n_lib_spectra, q_mode_ok, q_add_ok, cfg):
    """Collapse the (query spectrum x library spectrum) cosine matrix onto candidate structures."""
    n_q = cos.shape[0]
    # library spectra grouped by candidate structure (contiguous segments)
    order = np.argsort(lib_cand, kind='stable')
    counts = n_lib_spectra.astype(np.int64)
    starts = np.zeros(n_cand, np.int64)
    np.cumsum(counts[:-1], out=starts[1:])

    qmax = _segment_max(cos[:, order].astype(np.float64), starts, counts)     # (n_q, n_cand)
    best = qmax.max(axis=0) if n_q else np.zeros(n_cand)

    mo = q_mode_ok[:, order]
    usable = _segment_any(mo, starts, counts) if n_q else np.zeros((0, n_cand), bool)
    if not cfg.same_mode_only and n_q:
        usable = np.repeat((counts > 0)[None, :], n_q, axis=0)

    denom = usable.sum(axis=0) if n_q else np.zeros(n_cand)
    mean_qmax = np.where(denom > 0, (qmax * usable).sum(axis=0) / np.maximum(denom, 1), 0.0) \
        if n_q else np.zeros(n_cand)

    if cfg.agg == 'max':
        score = best
    elif cfg.agg == 'mean_qmax_all':
        score = qmax.mean(axis=0) if n_q else np.zeros(n_cand)
    elif cfg.agg.startswith('mean_top') and n_q:
        k = min(int(cfg.agg[len('mean_top'):]), n_q)
        srt = -np.sort(-qmax, axis=0)
        score = srt[:k].mean(axis=0)
    else:
        score = mean_qmax

    nm_best = (_segment_max(nmatch[:, order], starts, counts).max(axis=0)
               if n_q else np.zeros(n_cand, np.int32))
    matched_mode = usable.any(axis=0) if n_q else np.zeros(n_cand, bool)
    same_adduct = (_segment_any(q_add_ok[:, order], starts, counts).any(axis=0)
                   if n_q else np.zeros(n_cand, bool))

    base = base.copy()
    base['score'] = np.clip(score, 0.0, 1.0)
    base['n_lib_spectra'] = n_lib_spectra
    base['best_cosine'] = best
    base['mean_qmax'] = mean_qmax
    base['n_matched_best'] = nm_best
    base['matched_mode'] = matched_mode
    base['same_adduct'] = same_adduct
    return _finalise(base)


def _finalise(df):
    # ties (notably the many score-0 mass coincidences) are broken by how many spectra the
    # structure has in the library -- a weak but real "this compound gets measured" prior
    df = df.sort_values(['score', 'n_lib_spectra', 'ik14'], ascending=[False, False, True])
    return df.reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────
# Window helpers
# ─────────────────────────────────────────────────────────────────────

def _merge_windows(windows):
    w = np.asarray([wi for wi in windows if np.isfinite(wi[0]) and np.isfinite(wi[1])], dtype=np.float64)
    if len(w) == 0:
        return np.zeros(0), np.zeros(0)
    w = w[np.argsort(w[:, 0])]
    lo, hi = [w[0, 0]], [w[0, 1]]
    for a, b in w[1:]:
        if a <= hi[-1]:
            hi[-1] = max(hi[-1], b)
        else:
            lo.append(a)
            hi.append(b)
    return np.asarray(lo), np.asarray(hi)


def _in_windows(mass, lo, hi):
    if len(lo) == 0:
        return np.zeros(len(mass), bool)
    idx = np.searchsorted(lo, mass, side='right') - 1
    ok = idx >= 0
    out = np.zeros(len(mass), bool)
    out[ok] = mass[ok] <= hi[idx[ok]]
    return out & np.isfinite(mass)


# ─────────────────────────────────────────────────────────────────────
# Convenience: a whole test/validation set in one call
# ─────────────────────────────────────────────────────────────────────

def molecule_spectra(df, id_col='molecule_id'):
    """Group a test-style DataFrame into {molecule_id: [spectrum dicts]}."""
    out = {}
    for mid, g in df.groupby(id_col, sort=False):
        out[mid] = [{'mzs': r.ms2_mzs, 'intensities': r.ms2_normalized_intensities,
                     'precursor_mz': float(r.precursor_mz), 'adduct': r.adduct,
                     'ionization_mode': r.ionization_mode} for r in g.itertuples(index=False)]
    return out


def rank_molecules(train_path, structures_df, queries, config=None, ppm=DEFAULT_MASS_PPM,
                   exclude_libs=(), max_ppm_error=20.0, top_k=25, verbose=True, lib=None):
    """End-to-end: build the library for all queries, search each, return {mol_id: DataFrame}.

    queries: {molecule_id: [spectrum dicts]} as produced by `molecule_spectra`.
    """
    masses, windows = {}, []
    for mid, specs in queries.items():
        m = estimate_molecule_mass([s['precursor_mz'] for s in specs],
                                   [s['adduct'] for s in specs],
                                   [s['ionization_mode'] for s in specs])
        masses[mid] = m
        if np.isfinite(m):
            windows.append(mass_window(m, ppm))
    if lib is None:
        lib = SpectralLibrary.from_train(train_path, structures_df, windows,
                                         exclude_libs=exclude_libs, max_ppm_error=max_ppm_error,
                                         config=config, verbose=verbose)
    elif config is not None:
        lib.set_config(config)
    out = {}
    for mid, specs in queries.items():
        m = masses[mid]
        if not np.isfinite(m):
            out[mid] = pd.DataFrame(columns=['ik14', 'smiles', 'score'])
            continue
        lo, hi = mass_window(m, ppm)
        out[mid] = lib.search(specs, lo, hi).head(top_k) if top_k else lib.search(specs, lo, hi)
    return out, lib


In [ ]:
%%writefile casmi/candidates.py
"""Fast offline candidate-structure lookup by neutral monoisotopic mass.

The candidate databases (``coconut.parquet``, ``pubchem.parquet``) are parquet files
sorted ascending by the ``mass`` column and written with small row groups, so the
per-row-group min/max statistics in the parquet footer act as a coarse index: a
narrow mass window touches one or two row groups out of hundreds.

``CandidateDB.__init__`` reads only the footer (no data pages), which makes opening a
multi-gigabyte file instant and keeps resident memory at a few hundred kilobytes.

Pure python + numpy + pyarrow + pandas, so the same file runs locally and inside the
Kaggle notebook with the parquet files in a read-only dataset directory.

    db = CandidateDB('/kaggle/input/casmi-db/pubchem.parquet', 'pubchem')
    df = db.query(302.04265 * (1 - 1e-5), 302.04265 * (1 + 1e-5))
    dfs = db.query_many([(lo, hi) for lo, hi in windows], columns=['smiles', 'mass'])
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

__all__ = ["CandidateDB", "MASS_COLUMN"]

MASS_COLUMN = "mass"


class CandidateDB:
    """Mass-window lookup over one candidate parquet file.

    Parameters
    ----------
    path : str
        Path to the parquet file (``coconut.parquet`` / ``pubchem.parquet``).
    name : str
        Value placed in the ``source`` column of every returned frame.
    """

    def __init__(self, path, name):
        self.path = str(path)
        self.name = str(name)
        try:
            self._pf = pq.ParquetFile(self.path, memory_map=True)
        except Exception:
            self._pf = pq.ParquetFile(self.path)

        md = self._pf.metadata
        self.num_rows = md.num_rows
        self.num_row_groups = md.num_row_groups
        self.columns = list(self._pf.schema_arrow.names)
        if MASS_COLUMN not in self.columns:
            raise ValueError(f"{self.path} has no '{MASS_COLUMN}' column")

        self._rg_rows = np.array(
            [md.row_group(i).num_rows for i in range(md.num_row_groups)], dtype=np.int64
        )
        self._rg_min, self._rg_max = self._mass_index(md)
        # Sorted-by-mass files let us binary-search the row-group index instead of
        # scanning it; verified once here rather than assumed.
        self._rg_sorted = bool(
            np.all(np.diff(self._rg_min) >= 0) and np.all(np.diff(self._rg_max) >= 0)
        )

    # ── row-group mass index ──────────────────────────────────────────────
    def _mass_index(self, md):
        """Per-row-group (min, max) of `mass`, from footer statistics when present."""
        n = md.num_row_groups
        lo = np.empty(n, dtype=np.float64)
        hi = np.empty(n, dtype=np.float64)
        col_idx = None
        for j in range(md.num_columns):
            if md.row_group(0).column(j).path_in_schema == MASS_COLUMN:
                col_idx = j
                break
        ok = col_idx is not None
        if ok:
            for i in range(n):
                st = md.row_group(i).column(col_idx).statistics
                if st is None or not st.has_min_max:
                    ok = False
                    break
                lo[i] = st.min
                hi[i] = st.max
        if ok:
            self.stats_source = "footer"
            return lo, hi
        # Fallback: one pass over just the mass column to build the index ourselves.
        self.stats_source = "scan"
        mass = self._pf.read(columns=[MASS_COLUMN]).column(MASS_COLUMN).to_numpy(
            zero_copy_only=False
        )
        start = 0
        for i, k in enumerate(self._rg_rows):
            chunk = mass[start:start + k]
            lo[i] = chunk.min() if k else np.inf
            hi[i] = chunk.max() if k else -np.inf
            start += k
        del mass
        return lo, hi

    def _row_groups_for(self, mass_lo, mass_hi):
        """Indices of the row groups whose [min, max] overlaps [mass_lo, mass_hi]."""
        if mass_hi < mass_lo:
            return np.empty(0, dtype=np.int64)
        if self._rg_sorted:
            first = int(np.searchsorted(self._rg_max, mass_lo, side="left"))
            last = int(np.searchsorted(self._rg_min, mass_hi, side="right"))
            if last <= first:
                return np.empty(0, dtype=np.int64)
            return np.arange(first, last, dtype=np.int64)
        return np.nonzero((self._rg_max >= mass_lo) & (self._rg_min <= mass_hi))[0]

    # ── reading ───────────────────────────────────────────────────────────
    def _resolve_columns(self, columns):
        """-> (columns to read from parquet, columns to return)."""
        if columns is None:
            want = list(self.columns)
        else:
            want = list(columns)
            missing = [c for c in want if c not in self.columns and c != "source"]
            if missing:
                raise KeyError(f"unknown column(s) {missing}; have {self.columns}")
            want = [c for c in want if c != "source"]
        read = want if MASS_COLUMN in want else want + [MASS_COLUMN]
        return read, want

    @staticmethod
    def _slice(tbl, mass_lo, mass_hi):
        """Rows of `tbl` with mass_lo <= mass <= mass_hi (table is mass-sorted inside a row group)."""
        mass = tbl.column(MASS_COLUMN).to_numpy(zero_copy_only=False)
        n = len(mass)
        if n == 0:
            return tbl.slice(0, 0)
        if mass[0] <= mass[-1] and (n < 3 or np.all(np.diff(mass) >= 0)):
            a = int(np.searchsorted(mass, mass_lo, side="left"))
            b = int(np.searchsorted(mass, mass_hi, side="right"))
            return tbl.slice(a, max(b - a, 0))
        sel = np.nonzero((mass >= mass_lo) & (mass <= mass_hi))[0]
        return tbl.take(pa.array(sel))

    def _finish(self, pieces, want):
        if pieces:
            tbl = pa.concat_tables(pieces) if len(pieces) > 1 else pieces[0]
        else:
            tbl = self._pf.schema_arrow.empty_table()
        df = tbl.select([c for c in want if c in tbl.schema.names]).to_pandas()
        df["source"] = self.name
        return df.reset_index(drop=True)

    # ── public API ────────────────────────────────────────────────────────
    def query(self, mass_lo, mass_hi, columns=None):
        """All rows with ``mass_lo <= mass <= mass_hi``, plus a ``source`` column."""
        read_cols, want = self._resolve_columns(columns)
        pieces = []
        for i in self._row_groups_for(mass_lo, mass_hi):
            part = self._slice(self._pf.read_row_group(int(i), columns=read_cols),
                               mass_lo, mass_hi)
            if part.num_rows:
                pieces.append(part)
        return self._finish(pieces, want)

    def query_many(self, windows, columns=None):
        """``query`` for many windows, reading every needed row group exactly once.

        Windows are grouped by row group, so 1500 narrow windows cost at most 1500
        row-group reads instead of one full pass per window.
        """
        read_cols, want = self._resolve_columns(columns)
        windows = list(windows)
        rg_to_windows: dict[int, list[int]] = {}
        per_window: list[list] = [[] for _ in windows]
        for w, (lo, hi) in enumerate(windows):
            for i in self._row_groups_for(lo, hi):
                rg_to_windows.setdefault(int(i), []).append(w)
        for i in sorted(rg_to_windows):
            tbl = self._pf.read_row_group(i, columns=read_cols)
            for w in rg_to_windows[i]:
                lo, hi = windows[w]
                part = self._slice(tbl, lo, hi)
                if part.num_rows:
                    per_window[w].append(part)
            del tbl
        return [self._finish(pieces, want) for pieces in per_window]

    # ── convenience ───────────────────────────────────────────────────────
    def query_ppm(self, mass, ppm=10.0, columns=None):
        d = mass * ppm * 1e-6
        return self.query(mass - d, mass + d, columns=columns)

    def counts_many(self, windows):
        """Number of rows in each window, without materialising the rows."""
        return [len(df) for df in self.query_many(windows, columns=[MASS_COLUMN])]

    def __repr__(self):
        return (f"CandidateDB(name={self.name!r}, rows={self.num_rows:,}, "
                f"row_groups={self.num_row_groups}, mass="
                f"{self._rg_min.min():.3f}..{self._rg_max.max():.3f}, "
                f"stats={self.stats_source})")


In [ ]:
%%writefile casmi/pipeline.py
"""
End-to-end inference: test spectra -> up to 25 ranked SMILES per molecule.

Candidate sources (merged per molecule on the InChIKey first block, all restricted to the adduct-aware mass window):
  1. training structures whose library spectra resemble the query    (class 1: public spectra exist)
  2. COCONUT and PubChem structures in the mass window                (class 2: known structure, no spectra)
  3. de novo samples from the decoder whose mass matches              (class 3: novel structure)

Every candidate gets the same features (library similarity, predicted-fingerprint cosine, decoder
log-likelihood, source flags) and one linear ranker orders them. The same code path is used for local
validation and for the Kaggle run; validation only adds per-molecule exclusions to simulate the classes.
"""

import os
import json
import time
import numpy as np
import pandas as pd
from multiprocessing import Pool

from casmi.common import (
    molecule_neutral_mass, ppm_window, mol_from_smiles, morgan_bits, exact_mass, flat_canonical_smiles,
    plain_key, metric_key, _tautomer_enumerator, FP_BITS,
)
from rdkit import Chem

DEFAULT_CFG = dict(
    ppm_tol=10.0,            # candidate retrieval window around the calibrated neutral mass
    denovo_ppm=12.0,         # a generated structure must match the precursor mass to be kept
    denovo_samples=64,       # samples per spectrum
    denovo_temperature=1.0,
    max_lib=200, max_coconut=150, max_pubchem=150, max_denovo=100,   # shortlist sizes sent to the decoder
    n_proc=4,
    n_out=25,
)

FEATURES = ['fp_cos', 'fp_rank', 'll_rel', 'll_sqrt', 'll_tok', 'll_rank', 'lib_score', 'lib_hi', 'in_lib', 'in_coconut',
            'in_pubchem', 'db_only_pubchem', 'denovo_frac', 'denovo_any']


def add_features(t, n_samples):
    """Ranking features for one molecule's candidate table (needs fp_cos, ll, n_tok, lib_score, source flags, denovo_cnt)."""
    t['ll_rel'] = (t.ll - t.ll.max()).clip(lower=-40.0)
    t['ll_sqrt'] = -np.sqrt(-t.ll_rel)
    t['ll_tok'] = (t.ll / np.maximum(t.n_tok, 1)).clip(lower=-4.0)
    t['ll_rank'] = -np.log1p(t.ll.rank(ascending=False, method='min').values - 1)
    t['fp_rank'] = -np.log1p(t.fp_cos.rank(ascending=False, method='min').values - 1)
    t['lib_hi'] = (t.lib_score - 0.4).clip(lower=0.0)
    t['db_only_pubchem'] = ((t.in_pubchem == 1) & (t.in_coconut == 0) & (t.in_lib == 0)).astype(int)
    t['denovo_frac'] = t.denovo_cnt / n_samples
    t['denovo_any'] = (t.denovo_cnt > 0).astype(int)
    return add_group_features(t)


RANKER_FEATURES = FEATURES + ['lib_top', 'lib_margin_top', 'lib_hi7', 'lib_hi9', 'lib_conf', 'analog_sim', 'analog_tani', 'log_nspec']


def add_group_features(t):
    """Library-confidence features: a high similarity that also beats the runner-up by a clear margin is near-certain."""
    lib = t.lib_score.values.astype(np.float64)
    order = np.sort(lib)[::-1]
    best = order[0] if len(order) else 0.0
    second = order[1] if len(order) > 1 else 0.0
    top = (lib >= best) & (lib > 0)
    margin = np.where(top, best - second, 0.0)
    t = t.copy()
    t['lib_top'] = top.astype(int)
    t['lib_margin_top'] = margin
    t['lib_hi7'] = np.clip(lib - 0.7, 0, None)
    t['lib_hi9'] = np.clip(lib - 0.85, 0, None)
    t['lib_conf'] = top * lib * np.minimum(margin / 0.3, 1.0)
    # Analog evidence: if a library compound matches the spectra well but is not the answer, the answer is usually a
    # close structural relative of it (isomers give near-identical spectra). Tanimoto to the best library hits.
    analog_sim, analog_tani = np.zeros(len(t)), np.zeros(len(t))
    if 'fp_bits' in t.columns and len(t):
        bits = [set(b.tolist()) if b is not None else set() for b in t.fp_bits]
        hits = [j for j in np.argsort(-lib)[:3] if lib[j] > 0.3]
        for i, a in enumerate(bits):
            for rank, j in enumerate(hits):
                if j == i or not a or not bits[j]:
                    continue
                tani = len(a & bits[j]) / len(a | bits[j])
                analog_sim[i] = max(analog_sim[i], lib[j] * tani)
                if analog_tani[i] == 0.0:
                    analog_tani[i] = tani                      # similarity to the best hit that is not the candidate itself
    t['analog_sim'], t['analog_tani'] = analog_sim, analog_tani
    t['log_nspec'] = np.log1p(t.n_spectra.values.astype(np.float64)) if 'n_spectra' in t.columns else 0.0
    return t


def log(*a):
    print(time.strftime('%H:%M:%S'), *a, flush=True)


def group_spectra(df):
    """test.parquet-style frame -> {molecule_id: [spectrum dict, ...]} preserving first-appearance order."""
    groups = {}
    for row in df.itertuples(index=False):
        groups.setdefault(row.molecule_id, []).append(dict(
            mzs=row.ms2_mzs, intensities=row.ms2_normalized_intensities, precursor_mz=float(row.precursor_mz),
            adduct=row.adduct, ionization_mode=row.ionization_mode,
            collision_energy_ev=getattr(row, 'collision_energy_ev', None)))
    return groups


# ── worker functions (module level so they pickle) ──────────────────

def _fp_worker(smiles):
    return morgan_bits(smiles)


def _canon_worker(smiles):
    """-> (tautomer-canonical flat SMILES, metric key); (None, None) if invalid."""
    mol = mol_from_smiles(smiles)
    if mol is None:
        return (None, None)
    try:
        if mol.GetNumHeavyAtoms() <= 120:
            mol = _tautomer_enumerator().Canonicalize(mol)
        smi = Chem.MolToSmiles(mol, isomericSmiles=False)
        return (smi, plain_key(mol))
    except Exception:
        return (flat_canonical_smiles(smiles), plain_key(smiles))


def _denovo_worker(args):
    smiles, mass, ppm = args
    mol = mol_from_smiles(smiles)
    if mol is None or '.' in smiles:
        return None
    m = exact_mass(mol)
    if not np.isfinite(m) or abs(m - mass) / mass * 1e6 > ppm:
        return None
    return (Chem.MolToSmiles(mol, isomericSmiles=False), plain_key(mol))


def _pool_map(fn, items, n_proc, chunksize=256):
    if len(items) == 0:
        return []
    if n_proc <= 1 or len(items) < 2000:
        return [fn(x) for x in items]
    with Pool(n_proc) as pool:
        return pool.map(fn, items, chunksize=chunksize)


def fp_cosine(pred, bit_lists):
    """Cosine between a predicted probability vector [4096] and binary fingerprints given as on-bit index arrays."""
    norm = float(np.linalg.norm(pred)) + 1e-9
    out = np.zeros(len(bit_lists), np.float32)
    for i, bits in enumerate(bit_lists):
        if bits is not None and len(bits):
            out[i] = pred[bits].sum() / (norm * np.sqrt(len(bits)))
    return out


class Pipeline:
    def __init__(self, scorer, structures, library=None, coconut=None, pubchem=None, ranker=None, cfg=None):
        self.scorer, self.library, self.coconut, self.pubchem = scorer, library, coconut, pubchem
        self.cfg = {**DEFAULT_CFG, **(cfg or {})}
        self.ranker = ranker                      # dict(features=[...], coef=[...], intercept=float) or None
        st = structures.sort_values('mass').reset_index(drop=True)
        self.st, self.st_mass = st, st.mass.values
        self._model_cache, self._fp_cache, self._canon_cache = {}, {}, {}   # reused across runs (validation scenarios)

    # ── stage A: model passes that need only the spectra ──
    def _model_stage(self, groups, masses):
        cfg, out = self.cfg, {}
        jobs = []
        for mid, spectra in groups.items():
            states, mask = self.scorer.encode(spectra)
            fp = self.scorer.fingerprint(states, mask)
            samples = self.scorer.sample(states, mask, n_samples=cfg['denovo_samples'], temperature=cfg['denovo_temperature']) \
                if cfg['denovo_samples'] > 0 else []
            out[mid] = dict(fp=fp, n_samples=max(1, len(spectra) * cfg['denovo_samples']))
            best = {}
            for smi, lp in samples:
                cnt, mx = best.get(smi, (0, -np.inf))
                best[smi] = (cnt + 1, max(mx, lp))
            out[mid]['raw'] = best
            jobs += [(smi, masses[mid], cfg['denovo_ppm']) for smi in best]
        checked = _pool_map(_denovo_worker, jobs, cfg['n_proc'])
        pos = 0
        for mid in groups:
            agg = {}
            for smi, (cnt, lp) in out[mid].pop('raw').items():
                res = checked[pos]; pos += 1
                if res is None or res[1] is None:
                    continue
                canon, key = res
                c0, l0, _ = agg.get(key, (0, -np.inf, canon))
                agg[key] = (c0 + cnt, max(l0, lp), canon)
            out[mid]['denovo'] = agg
        return out

    # ── stage B: retrieval ──
    def _library_candidates(self, mid, spectra, lo, hi, exclude):
        i0, i1 = np.searchsorted(self.st_mass, lo), np.searchsorted(self.st_mass, hi, side='right')
        cand = self.st.iloc[i0:i1][['ik14', 'smiles', 'fp_bits', 'mkey', 'n_spectra']].copy()
        if exclude.get('ik14_lib'):
            cand = cand[~cand.ik14.isin(exclude['ik14_lib'])]
        cand['lib_score'] = 0.0
        if self.library is not None and len(cand):
            res = self.library.search(spectra, lo, hi, exclude_libs=tuple(exclude.get('libs', ())),
                                      exclude_ik14=tuple(exclude.get('ik14_lib', ())))
            if len(res):
                cand['lib_score'] = cand.ik14.map(dict(zip(res.ik14, res.score))).fillna(0.0).values
        return cand

    def run(self, groups, exclusions=None, return_features=False):
        cfg, exclusions = self.cfg, exclusions or {}
        mids = list(groups)
        masses = {m: molecule_neutral_mass([s['precursor_mz'] for s in groups[m]], [s['adduct'] for s in groups[m]]) for m in mids}
        windows = {m: ppm_window(masses[m], cfg['ppm_tol']) for m in mids}
        log(f'{len(mids)} molecules | model stage (fingerprints + de novo sampling)...')
        cache_key = (tuple(mids), cfg['denovo_samples'], cfg['denovo_temperature'])
        if cache_key not in self._model_cache:
            self._model_cache = {cache_key: self._model_stage(groups, masses)}
        model_out = self._model_cache[cache_key]

        log('retrieval stage...')
        db_frames = {}
        for name, db in (('coconut', self.coconut), ('pubchem', self.pubchem)):
            if db is not None:
                frames = db.query_many([windows[m] for m in mids], columns=['smiles', 'ik14'])
                db_frames[name] = dict(zip(mids, frames))

        tables = {}
        for mid in mids:
            ex = exclusions.get(mid, {})
            lo, hi = windows[mid]
            lib = self._library_candidates(mid, groups[mid], lo, hi, ex)
            rows = {k: dict(ik14=k, smiles=s, fp_bits=b, lib_score=l, in_lib=1, in_coconut=0, in_pubchem=0, denovo_cnt=0,
                            needs_canon=False, mkey=mk, n_spectra=ns)
                    for k, s, b, l, mk, ns in zip(lib.ik14, lib.smiles, lib.fp_bits, lib.lib_score, lib.mkey, lib.n_spectra)}
            for name in ('coconut', 'pubchem'):
                if name not in db_frames:
                    continue
                f = db_frames[name][mid]
                drop = ex.get('ik14_db') or ()
                for k, s in zip(f.ik14.values, f.smiles.values):
                    if k in drop:
                        continue
                    r = rows.get(k)
                    if r is None:
                        r = rows[k] = dict(ik14=k, smiles=s, fp_bits=None, lib_score=0.0, in_lib=0, in_coconut=0, in_pubchem=0,
                                           denovo_cnt=0, needs_canon=True, mkey=k, n_spectra=0)
                    r['in_' + name] = 1
            for k, (cnt, lp, canon) in model_out[mid]['denovo'].items():
                r = rows.get(k)
                if r is None:
                    r = rows[k] = dict(ik14=k, smiles=canon, fp_bits=None, lib_score=0.0, in_lib=0, in_coconut=0, in_pubchem=0,
                                       denovo_cnt=0, needs_canon=False, mkey=k, n_spectra=0)
                r['denovo_cnt'] = cnt
            tables[mid] = pd.DataFrame(list(rows.values())) if rows else pd.DataFrame(
                columns=['ik14', 'smiles', 'fp_bits', 'lib_score', 'in_lib', 'in_coconut', 'in_pubchem', 'denovo_cnt', 'needs_canon', 'mkey', 'n_spectra'])

        n_total = sum(len(t) for t in tables.values())
        log(f'fingerprint stage: {n_total:,} candidates...')
        todo = sorted({s for t in tables.values() for s, b in zip(t.smiles, t.fp_bits) if b is None} - self._fp_cache.keys())
        self._fp_cache.update(zip(todo, _pool_map(_fp_worker, todo, cfg['n_proc'], chunksize=2000)))
        bits = self._fp_cache
        for mid, t in tables.items():
            if len(t) == 0:
                continue
            t['fp_bits'] = [b if b is not None else bits.get(s) for s, b in zip(t.smiles, t.fp_bits)]
            t['fp_cos'] = fp_cosine(model_out[mid]['fp'], t.fp_bits.tolist())

        log('shortlist + canonicalisation stage...')
        for mid, t in tables.items():
            if len(t) == 0:
                continue
            t['log_ncand'] = np.log1p(len(t))
            keep = np.zeros(len(t), bool)
            pre = t.fp_cos.values + t.lib_score.values
            for flag, cap in (('in_lib', cfg['max_lib']), ('in_coconut', cfg['max_coconut']), ('in_pubchem', cfg['max_pubchem'])):
                idx = np.flatnonzero(t[flag].values == 1)
                keep[idx[np.argsort(-pre[idx])[:cap]]] = True
            idx = np.flatnonzero(t.denovo_cnt.values > 0)
            keep[idx[np.argsort(-t.denovo_cnt.values[idx])[:cfg['max_denovo']]]] = True
            tables[mid] = t[keep].reset_index(drop=True)
        todo = sorted({s for t in tables.values() if len(t) for s, n in zip(t.smiles, t.needs_canon) if n} - self._canon_cache.keys())
        self._canon_cache.update(zip(todo, _pool_map(_canon_worker, todo, cfg['n_proc'], chunksize=64)))
        canon = self._canon_cache
        for mid, t in tables.items():
            if len(t) == 0:
                continue
            t['score_smiles'] = [(canon[s][0] or s) if n else s for s, n in zip(t.smiles, t.needs_canon)]
            t['mkey'] = [(canon[s][1] or k) if n else k for s, n, k in zip(t.smiles, t.needs_canon, t.mkey)]

        log('decoder likelihood stage...')
        for mid, t in tables.items():
            if len(t) == 0:
                continue
            states, mask = self.scorer.encode(groups[mid])
            ll, n_tok = self.scorer.loglik(states, mask, t.score_smiles.tolist())
            finite = np.isfinite(ll)
            floor = (ll[finite].min() - 5.0) if finite.any() else -100.0
            t['ll'] = np.where(finite, ll, floor)
            t['n_tok'] = n_tok
            tables[mid] = t = add_features(t, model_out[mid]['n_samples'])

        log('ranking stage...')
        results = {}
        for mid, t in tables.items():
            if len(t) == 0:
                results[mid] = []
                continue
            t['final'] = self.rank_score(t)
            t.sort_values('final', ascending=False, inplace=True, kind='stable')
            t.reset_index(drop=True, inplace=True)
        # dedupe the head of each list on the real metric key (tautomer-canonical)
        head = {mid: t.score_smiles.head(3 * cfg['n_out']).tolist() for mid, t in tables.items() if len(t)}
        todo = sorted({s for v in head.values() for s in v} - self._canon_cache.keys())
        self._canon_cache.update(zip(todo, _pool_map(_canon_worker, todo, cfg['n_proc'], chunksize=64)))
        keyed = self._canon_cache
        for mid, smiles in head.items():
            seen, out = set(), []
            for s in smiles:
                canon_smi, key = keyed.get(s, (None, None))
                if canon_smi is None or key in seen:
                    continue
                seen.add(key); out.append(canon_smi)
                if len(out) == cfg['n_out']:
                    break
            results[mid] = out
        log('done')
        return (results, tables) if return_features else results

    def rank_score(self, t):
        if self.ranker is None:     # untuned fallback: library evidence first, then the two model scores
            return 2.0 * t.lib_score.values + t.fp_cos.values + 0.05 * t.ll_rel.values + 0.3 * t.in_lib.values \
                + 0.15 * t.in_coconut.values
        r = self.ranker
        x = t[r['features']].values.astype(np.float64)
        x = (x - np.asarray(r['mean'])) / np.asarray(r['scale'])
        if 'coef' in r:
            return x @ np.asarray(r['coef'])
        w = {k: np.asarray(v) for k, v in r['weights'].items()}
        if r.get('hidden'):
            h = np.tanh(x @ w['0.weight'].T + w['0.bias'])
            return (h @ w['2.weight'].T + w['2.bias'])[:, 0]
        return (x @ w['weight'].T)[:, 0]


def write_submission(results, sample_submission_path, out_path, fallback='CCO'):
    sub = pd.read_csv(sample_submission_path)
    sub['smiles'] = [';'.join(results.get(m, [])[:25]) or fallback for m in sub.molecule_id]
    sub.to_csv(out_path, index=False)
    return sub


def load_ranker(path):
    if path and os.path.exists(path):
        with open(path) as fh:
            return json.load(fh)
    return None


In [ ]:
%%writefile casmi/run_test.py
"""
Produce submission.csv for a test.parquet. Same entry point locally and inside the Kaggle notebook.

    python -m casmi.run_test                                   # local: ./test.parquet -> output/submission.csv
"""
import os
import argparse
import pandas as pd

from casmi.common import molecule_neutral_mass, ppm_window
from casmi.pipeline import Pipeline, group_spectra, write_submission, load_ranker, log
from casmi.infer import ModelScorer

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def run(test_path, sample_submission_path, train_path, assets_dir, out_path, n_proc=4, denovo_samples=64, use_pubchem=True):
    from casmi.libsearch import SpectralLibrary
    from casmi.candidates import CandidateDB
    test = pd.read_parquet(test_path)
    groups = group_spectra(test)
    log(f'test: {len(test)} spectra, {len(groups)} molecules')
    structures = pd.read_parquet(os.path.join(assets_dir, 'structures.parquet'))
    structures = structures[structures.mkey.notna()]
    scorer = ModelScorer(os.path.join(assets_dir, 'model.pt'), os.path.join(assets_dir, 'vocab.json'))
    log(f'model on {scorer.device} (bf16={scorer.amp})')
    windows = [ppm_window(molecule_neutral_mass([s['precursor_mz'] for s in g], [s['adduct'] for s in g]), 10.0) for g in groups.values()]
    library = SpectralLibrary.from_train(train_path, structures, windows)
    coconut = CandidateDB(os.path.join(assets_dir, 'coconut.parquet'), 'coconut')
    pub_path = os.path.join(assets_dir, 'pubchem.parquet')
    pubchem = CandidateDB(pub_path, 'pubchem') if use_pubchem and os.path.exists(pub_path) else None
    ranker = load_ranker(os.path.join(assets_dir, 'ranker.json'))
    pipe = Pipeline(scorer, structures, library, coconut, pubchem, ranker=ranker, cfg=dict(n_proc=n_proc, denovo_samples=denovo_samples))
    results = pipe.run(groups)
    sub = write_submission(results, sample_submission_path, out_path)
    n = sub.smiles.str.count(';') + 1
    log(f'wrote {len(sub)} rows to {out_path} | guesses per molecule: median {int(n.median())}, min {n.min()}, max {n.max()} '
        f'| fallback rows {(sub.smiles == "CCO").sum()}')
    return sub


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--test', default=os.path.join(ROOT, 'test.parquet'))
    ap.add_argument('--sample', default=os.path.join(ROOT, 'sample_submission.csv'))
    ap.add_argument('--train', default=os.path.join(ROOT, 'train.parquet'))
    ap.add_argument('--assets', default=os.path.join(ROOT, 'kaggle_assets'))
    ap.add_argument('--out', default=os.path.join(ROOT, 'output', 'submission.csv'))
    ap.add_argument('--n-proc', type=int, default=8)
    a = ap.parse_args()
    run(a.test, a.sample, a.train, a.assets, a.out, n_proc=a.n_proc)


In [ ]:
from casmi.run_test import run

submission = run(
    test_path=os.path.join(COMP_DIR, 'test.parquet'),
    sample_submission_path=os.path.join(COMP_DIR, 'sample_submission.csv'),
    train_path=os.path.join(COMP_DIR, 'train.parquet'),
    assets_dir=ASSETS_DIR,
    out_path='/kaggle/working/submission.csv',
    n_proc=4,
)
submission.head()